# Predictive Modeling

This notebook is a scaffold for single-asset predictive modeling workflows in the Pricing folder. It loads the shared Single Asset Profile parameters, downloads the asset history, engineers a compact 3-feature classifier, and estimates the probability that the close will be higher in 3 trading days.

Workflow map:
1. Import the modeling dependencies.
2. Load the shared Single Asset Profile parameters.
3. Download price history, engineer features, and create the 3-day up/down target.
4. Reserve a block for feature filtering.
5. Plot the forward 3-day return target.
6. Reserve a block for normalization before model training.
7. Train and evaluate the logistic regression classifier.
8. Score the latest observation and collect test-set probabilities.
9. Plot predicted probabilities against realized outcomes.

In [ ]:
# Block 1: Import the modeling dependencies and notebook setup.

import warnings
from pathlib import Path
import runpy
from IPython.display import display

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

In [ ]:
# Block 2: Load the shared Single Asset Profile parameters.

PARAMS_FILE_CANDIDATES = []
for base_path in (Path.cwd(), *Path.cwd().parents):
    PARAMS_FILE_CANDIDATES.extend([
        base_path / "params.py",
        base_path / "Notebooks" / "Single Asset Profile" / "params.py",
    ])

seen_params_paths = set()
for params_file in PARAMS_FILE_CANDIDATES:
    params_file = params_file.resolve()
    if params_file in seen_params_paths:
        continue
    seen_params_paths.add(params_file)
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
single_asset_params = notebook_params["get_single_asset_profile_params"]()
resolve_time_frame_map = notebook_params["resolve_time_frame_map"]

ticker_str = single_asset_params["ticker_str"]
interval = single_asset_params["interval"]
period = single_asset_params["period"]
risk_free_ticker = single_asset_params["risk_free_ticker"]
benchmark_tickers = list(single_asset_params["benchmark_tickers"])
trading_strategy = single_asset_params["trading_strategy"]
time_frame_map = resolve_time_frame_map(trading_strategy)

single_asset_params

In [ ]:
# Block 3: Download price history, engineer candidate features, and build the 3-day classification target.

price_frame = yf.Ticker(ticker_str).history(period=period, interval=interval).copy()
if price_frame.empty:
    raise ValueError(f"No price history returned for {ticker_str}.")

price_frame.index = pd.to_datetime(price_frame.index).tz_localize(None)
price_frame = price_frame.sort_index()

# `feature_window_max` is the adjustable parameter that controls how many
# rolling/lookback windows we create for each feature family.
feature_window_max = 21
feature_windows = list(range(1, feature_window_max + 1))

price_frame["return_1d"] = price_frame["Close"].pct_change()

return_feature_columns = []
for window in feature_windows:
    column_name = f"return_{window}d"
    if window == 1:
        price_frame[column_name] = price_frame["return_1d"]
    else:
        price_frame[column_name] = price_frame["Close"].pct_change(window)
    return_feature_columns.append(column_name)

volatility_feature_columns = []
for window in feature_windows:
    column_name = f"volatility_{window}d"
    rolling_std = price_frame["return_1d"].rolling(window).std(ddof=0)
    price_frame[column_name] = rolling_std * np.sqrt(252)
    volatility_feature_columns.append(column_name)

volume_change_feature_columns = []
for window in feature_windows:
    column_name = f"volume_change_{window}d"
    price_frame[column_name] = price_frame["Volume"].pct_change(window)
    volume_change_feature_columns.append(column_name)

sharpe_feature_columns = []
for window in feature_windows:
    column_name = f"sharpe_{window}d"
    rolling_mean = price_frame["return_1d"].rolling(window).mean()
    rolling_std = price_frame["return_1d"].rolling(window).std(ddof=0)
    sharpe_series = (rolling_mean / rolling_std.replace(0, np.nan)) * np.sqrt(252)
    price_frame[column_name] = sharpe_series.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    sharpe_feature_columns.append(column_name)

candidate_feature_columns = [
    *return_feature_columns,
    *volatility_feature_columns,
    *volume_change_feature_columns,
    *sharpe_feature_columns,
]

price_frame["close_3d_ahead"] = price_frame["Close"].shift(-3)
price_frame["next_return_3d"] = price_frame["close_3d_ahead"] / price_frame["Close"] - 1
price_frame["target_up_3d"] = np.where(
    price_frame["close_3d_ahead"].notna(),
    (price_frame["close_3d_ahead"] > price_frame["Close"]).astype(int),
    np.nan,
    )

prediction_frame = price_frame[["Close", *candidate_feature_columns]].replace([np.inf, -np.inf], np.nan).dropna().copy()

model_frame = price_frame[["Close", *candidate_feature_columns, "next_return_3d", "target_up_3d"]].replace([np.inf, -np.inf], np.nan).dropna().copy()

model_frame.tail()

In [ ]:
# Block 4: Filter candidate features for low variance and multicollinearity.

# Use the training slice only so the filtering step does not peek into the test set.
feature_filter_split_index = max(int(len(model_frame) * 0.8), 1)
feature_filter_train = model_frame.iloc[:feature_filter_split_index][candidate_feature_columns].copy()

# Adjustable filtering thresholds.
low_variance_threshold = 1e-6
correlation_threshold = 0.95

feature_variances = feature_filter_train.var(ddof=0)
low_variance_dropped_columns = feature_variances.loc[
    feature_variances <= low_variance_threshold
].index.tolist()

variance_kept_columns = [
    column for column in candidate_feature_columns if column not in low_variance_dropped_columns
]

if variance_kept_columns:
    correlation_matrix = feature_filter_train[variance_kept_columns].corr().abs()
    upper_triangle = correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )
    multicollinearity_dropped_columns = [
        column for column in upper_triangle.columns if (upper_triangle[column] > correlation_threshold).any()
    ]
else:
    correlation_matrix = pd.DataFrame()
    multicollinearity_dropped_columns = []

filtered_candidate_feature_columns = [
    column
    for column in variance_kept_columns
    if column not in multicollinearity_dropped_columns
]

feature_filter_summary = pd.DataFrame(
    [
        {"metric": "candidate_features_before_filtering", "value": len(candidate_feature_columns)},
        {"metric": "low_variance_dropped", "value": len(low_variance_dropped_columns)},
        {"metric": "multicollinearity_dropped", "value": len(multicollinearity_dropped_columns)},
        {"metric": "candidate_features_after_filtering", "value": len(filtered_candidate_feature_columns)},
        {"metric": "low_variance_threshold", "value": low_variance_threshold},
        {"metric": "correlation_threshold", "value": correlation_threshold},
    ]
)

feature_filter_before_after = pd.DataFrame(
    {
        "before_filtering": pd.Series(candidate_feature_columns, dtype="object"),
        "after_filtering": pd.Series(filtered_candidate_feature_columns, dtype="object"),
    }
)

feature_filter_details = pd.DataFrame(
    {
        "low_variance_dropped": pd.Series(low_variance_dropped_columns, dtype="object"),
        "multicollinearity_dropped": pd.Series(multicollinearity_dropped_columns, dtype="object"),
        "filtered_candidate_features": pd.Series(filtered_candidate_feature_columns, dtype="object"),
    }
)

display(feature_filter_summary)
display(feature_filter_before_after)
feature_filter_details.head(25)


In [ ]:
# Block 5: Plot the forward 3-day return target used to derive the classifier label.

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=model_frame.index,
        y=model_frame["next_return_3d"] * 100,
        name="Forward 3-day return",
        line={"color": "#60a5fa", "width": 2},
    )
)
fig.add_hline(y=0, line_dash="dot", line_color="#cbd5e1")
fig.update_layout(
    title=f"{ticker_str} forward 3-day return target",
    template="plotly_dark",
    xaxis_title="Date",
    yaxis_title="Forward return (%)",
    height=500,
)
fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# Block 6: Normalization placeholder.

# We'll add the model normalization/preprocessing logic here later.


In [ ]:
# Block 7: Train the logistic classifier and evaluate it on a time-ordered test split.

# Start with a compact subset from the filtered candidate feature pool.
# We use the 3-day window here because it matches the current target horizon.
starter_feature_window = min(3, feature_window_max)
preferred_feature_columns = [
    f"return_{starter_feature_window}d",
    f"volatility_{starter_feature_window}d",
    f"volume_change_{starter_feature_window}d",
    f"sharpe_{starter_feature_window}d",
]
feature_columns = [
    column for column in preferred_feature_columns if column in filtered_candidate_feature_columns
]
if len(feature_columns) < len(preferred_feature_columns):
    fallback_feature_columns = [
        column for column in filtered_candidate_feature_columns if column not in feature_columns
    ]
    feature_columns.extend(fallback_feature_columns[: len(preferred_feature_columns) - len(feature_columns)])

if not feature_columns:
    raise ValueError("Feature filtering removed every candidate feature. Relax the filtering thresholds.")

thresholds = [0.50, 0.525, 0.55, 0.575, 0.60]

split_index = max(int(len(model_frame) * 0.8), 1)
train_frame = model_frame.iloc[:split_index].copy()
test_frame = model_frame.iloc[split_index:].copy()

if train_frame.empty or test_frame.empty:
    raise ValueError("Not enough observations to create a time-ordered train/test split.")

X_train = train_frame[feature_columns]
y_train = train_frame["target_up_3d"].astype(int)
X_test = test_frame[feature_columns]
y_test = test_frame["target_up_3d"].astype(int)

logit_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)
logit_model.fit(X_train, y_train)

test_probabilities = logit_model.predict_proba(X_test)[:, 1]
test_roc_auc = roc_auc_score(y_test, test_probabilities) if y_test.nunique() > 1 else np.nan
test_brier_score = brier_score_loss(y_test, test_probabilities)
test_base_rate = y_test.mean()

threshold_predictions = {}
threshold_rows = []
for threshold in thresholds:
    current_predictions = (test_probabilities >= threshold).astype(int)
    threshold_predictions[threshold] = current_predictions
    threshold_rows.append(
        {
            "threshold_probability": threshold,
            "threshold_percent": threshold * 100,
            "test_accuracy": accuracy_score(y_test, current_predictions),
            "predicted_up_rate": current_predictions.mean(),
            "predicted_up_count": int(current_predictions.sum()),
        }
    )

default_threshold = 0.50
test_predictions = threshold_predictions[default_threshold]

evaluation = pd.Series(
    {
        "train_rows": len(train_frame),
        "test_rows": len(test_frame),
        "test_accuracy_at_default_threshold": accuracy_score(y_test, test_predictions),
        "test_roc_auc": test_roc_auc,
        "test_brier_score": test_brier_score,
        "test_base_rate": test_base_rate,
        "default_threshold_probability": default_threshold,
    },
    name="value",
).to_frame()

threshold_metrics = pd.DataFrame(threshold_rows)
threshold_metrics["threshold_label"] = threshold_metrics["threshold_percent"].map(lambda value: f"{value:.1f}%")
threshold_metrics["test_roc_auc"] = test_roc_auc
threshold_metrics["test_brier_score"] = test_brier_score
threshold_metrics["test_base_rate"] = test_base_rate
threshold_metrics = threshold_metrics[[
    "threshold_label",
    "threshold_probability",
    "test_accuracy",
    "predicted_up_rate",
    "predicted_up_count",
    "test_roc_auc",
    "test_brier_score",
    "test_base_rate",
]].round(
    {
        "threshold_probability": 3,
        "test_accuracy": 4,
        "predicted_up_rate": 4,
        "test_roc_auc": 4,
        "test_brier_score": 4,
        "test_base_rate": 4,
    }
)

display(evaluation)
threshold_metrics

In [ ]:
# Block 8: Score the latest observation and store the test-set probabilities for review.

latest_features = prediction_frame[feature_columns].iloc[[-1]]
latest_timestamp = prediction_frame.index[-1]
current_close = float(prediction_frame.loc[latest_timestamp, "Close"])
latest_probability = float(logit_model.predict_proba(latest_features)[0, 1])

probability_frame = test_frame[["Close", "next_return_3d", "target_up_3d"]].copy()
probability_frame["predicted_probability_up_3d"] = test_probabilities
for threshold, current_predictions in threshold_predictions.items():
    threshold_suffix = f"{threshold * 100:.1f}".replace(".", "_")
    probability_frame[f"predicted_up_{threshold_suffix}pct"] = current_predictions

latest_signal_summary = {
    "as_of_date": latest_timestamp,
    "current_close": round(current_close, 2),
    "probability_close_higher_in_3d": round(latest_probability * 100, 2),
}
for threshold in thresholds:
    threshold_suffix = f"{threshold * 100:.1f}".replace(".", "_")
    latest_signal_summary[f"predict_up_at_{threshold_suffix}pct"] = int(latest_probability >= threshold)

pd.DataFrame([latest_signal_summary]).set_index("as_of_date")

In [ ]:
# Block 9: Compare predicted probabilities with the realized 0/1 outcomes and threshold ladder.

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=probability_frame.index,
        y=probability_frame["predicted_probability_up_3d"] * 100,
        name="Predicted probability of higher close in 3 days",
        line={"color": "#34d399", "width": 2},
    )
)
fig.add_trace(
    go.Scatter(
        x=probability_frame.index,
        y=probability_frame["target_up_3d"] * 100,
        name="Realized outcome (0 or 100)",
        mode="markers",
        marker={"color": "#fbbf24", "size": 7, "opacity": 0.7},
    )
)
for threshold in thresholds:
    fig.add_hline(
        y=threshold * 100,
        line_dash="dot",
        line_color="#cbd5e1" if np.isclose(threshold, default_threshold) else "#64748b",
        opacity=0.65 if np.isclose(threshold, default_threshold) else 0.25,
    )
fig.update_layout(
    title=f"{ticker_str} logistic regression probability of a higher close in 3 trading days<br><sup>Decision thresholds shown: 50.0%, 52.5%, 55.0%, 57.5%, 60.0%</sup>",
    template="plotly_dark",
    xaxis_title="Date",
    yaxis_title="Probability (%)",
    height=500,
)
fig.show(config={"responsive": True, "displaylogo": False})